# Поиск объявлений услуг для Авито

## Задача и результат

Для каждого запроса нужно выбрать **до 50 объявлений** из корпуса. Правильными считаются объявления, которые пользователь выбрал по этому запросу. Результат — `answer.csv`: одна строка на запрос, два столбца `query_id` и `answer`.

Качество измеряется **Recall@50**. Если для запроса известно 3 правильных объявления, а поиск вернул 2 из них, оценка этого запроса равна `2 / 3`. Итоговая оценка — среднее по запросам. Порядок внутри 50 ответов не важен.

Мы последовательно проверим три подхода:

1. **Поиск по словам (TF-IDF)** — отправная точка для сравнения.
2. **Поиск по словам и частям слов с учётом города** — способ собрать больше подходящих кандидатов.
3. **Обучаемый отбор** — градиентный бустинг выбирает 50 объявлений из расширенного списка кандидатов.

Решение работает локально на CPU, без внешних API и скачивания нейросетей. Мы используем pandas, NumPy и scikit-learn. Сложность здесь оправдана только тогда, когда она помогает качеству на отдельной проверочной выборке.

### Как запускать

Положить три исходных Parquet-файла в `data/`, установить зависимости из `requirements.txt`, запустить Jupyter из папки проекта и выполнить ячейки сверху вниз. Для нового полного запуска перезапустить ядро. Данные и расчёты занимают заметный объём памяти; полный проверочный запуск занял около 35 минут. Прежняя версия сохранена в `baseline.ipynb`.

Дополнительные материалы: [разбор конструкций кода](CODE_GUIDE.md), [подробный отчёт с результатами](REPORT.md), [инструкция запуска](README.md).

## Версия с комментариями и документацией функций

Описания находятся **внутри функций**, сразу после строки `def`, в тройных кавычках. Например, найдите `def clean_text`: под ним описано, что функция принимает и возвращает. Короткие комментарии начинаются с `#`.

В этой версии сложные выражения развернуты в отдельные шаги. Выводы ячеек сохранены от прежнего полного эксперимента; после этого восстановления полный расчёт не запускался заново.


## 1. Библиотеки и настройки

`pandas` нужен для таблиц, `numpy` — для массивов чисел, `re` — для очистки текста, `json` — для сохранения результатов эксперимента. Число `42` фиксирует случайный выбор: при одинаковых данных и версиях библиотек разбиение повторяется.

Для первого воспроизводимого эксперимента используем до 2500 запросов для обучения, 500 для выбора варианта и 500 для окончательной проверки. Это ограничение времени расчёта, а не свойство алгоритма.

In [2]:
import json
import os
import re
import time

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
from threadpoolctl import threadpool_limits

SEED = 42
TRAIN_SIZE = 2500
DEV_SIZE = 500
TEST_SIZE = 500
TOP_K = 50
CHANNEL_SIZE = 100
BATCH_SIZE = 32

os.makedirs("outputs", exist_ok=True)
start_time = time.time()

## 2. Загрузка исходных данных

В `train` каждая строка — факт выбора объявления по запросу. В `items` находится корпус, среди которого разрешено искать. В `queries` находятся запросы, для которых нужно сформировать ответ.

Из `train` читаем только условия поиска и `item_id`: свойства объявлений берём из корпуса. Это экономит память. Идентификаторы остаются строками, чтобы не потерять ведущие нули.

In [3]:
search_columns = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

train = pd.read_parquet("data/train.parquet", columns=search_columns + ["item_id"])
queries = pd.read_parquet("data/benchmark_queries.parquet")
items = pd.read_parquet("data/benchmark_items.parquet")

print("Строк обучения:", len(train))
print("Запросов бенчмарка:", len(queries))
print("Объявлений в корпусе:", len(items))

Строк обучения: 497673
Запросов бенчмарка: 2452
Объявлений в корпусе: 189212


In [4]:
train.head(3)

,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_id
0,скупка телевизоров,652430,0,,114,e8b685dffe1a408e
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,92e1b0446f827b59
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Ви...",114,624846856ce81d69


In [5]:
queries.head(3)

,query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,70DfDUpwjxB4lzFd,перевозки владикавказ тбилиси,649820,0,,114
1,JTrdTaZJvSiLPkXj,обзвон по базе,107620,0,Вид услуги Деловые услуги,114
2,LZCZNoVG4AFUkVRJ,липоредукция подбородка,637640,0,"Вид услуги Красота, здоровье",114


In [6]:
items.head(3)

,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,Ремонт/выкуп компьют. и ноутбуков с выездом на...,57.0,5.0,500.000000000000000,2097573,42.047193961004901,631060,44.228180450924000,False,False,Вид услуги Компьютерная помощь Место оказания ...,111eb8b979577d79,"Выезд на дом в любое время, вплоть до 23:00\n\...",114
1,Обучение ребенка чтению,7.0,5.0,900.000000000000000,86453,49.627391820000000,631870,58.627609249999999,False,False,"Вид услуги Обучение, курсы Место оказания услу...",75fc8e10f5a66fc4,"Дорогие родители и маленькие книголюбы! \n\nЯ,...",114
2,Афрокудри 5+,14.0,5.0,1000.000000000000000,86467,40.936892000000000,628500,56.990307000000001,True,False,"Вид услуги Красота, здоровье Место оказания ус...",3f6ae81704565b9c,ВНИМАНИЕ Укладка !!!! 🥳\n\nАФРОкудри отличный ...,114


## 3. Проверка данных

Пустые описания допустимы — заменим их пустой строкой. Отсутствующий или повторяющийся идентификатор недопустим: тогда нельзя однозначно связать объявление с ответом.

`assert` останавливает выполнение, если проверка не прошла. `.all()` означает «условие выполнено для всех строк», `.is_unique` — «повторов нет».

In [7]:
assert queries["query_id"].notna().all(), "Есть пустые query_id"
assert queries["query_id"].is_unique, "Повторяются query_id"
assert (queries["query_id"].str.len() == 16).all(), "Неверная длина query_id"
assert items["item_id"].is_unique, "Повторяются item_id"
assert items["item_id"].str.fullmatch("[0-9a-f]{16}", na=False).all(), "Неверный item_id"

print("Типы полей запроса:")
print(queries.dtypes)
print("\nПропуски в корпусе:")
print(items.isna().sum())

Типы полей запроса:
query_id                       str
search_query                   str
search_location_id           int64
search_is_delivery_search    int32
search_infm_params_text        str
search_category              int64
dtype: object

Пропуски в корпусе:
item_title_raw                   0
item_rating_reviews_count    11615
item_rating                  17631
item_price                       0
item_microcat_id                 0
item_longitude                   1
item_location_id                 0
item_latitude                    1
item_is_phone_hidden             0
item_is_message_forbidden        0
item_infm_params_text            0
item_id                          0
item_description_raw            35
item_category_id                 0
dtype: int64


In [8]:
print("Уникальных текстов в train:", train["search_query"].nunique())
print("Повторов условий в benchmark:", queries.duplicated(search_columns).sum())

known_queries = queries["search_query"].isin(train["search_query"])
known_items = train["item_id"].isin(items["item_id"])

print("Текст запроса был в train (%):", round(known_queries.mean() * 100, 2))
print("Строк train с объявлением из корпуса (%):", round(known_items.mean() * 100, 2))

Уникальных текстов в train: 74529
Повторов условий в benchmark: 0


Текст запроса был в train (%): 36.99
Строк train с объявлением из корпуса (%): 6.63


**Как читать эти числа.** Совпадение текста запроса с train не означает, что известны все ответы бенчмарка. Локация, категория и фильтры могут отличаться. Пересечение объявлений показывает, какую часть train можно использовать для проверки именно на данном корпусе. Нельзя считать все объявления, которые не были выбраны, заведомо плохими: пользователь мог просто их не увидеть.

## 4. Очистка текста

Теперь `clean_text` принимает одну строку. Поэтому запрос можно обработать простым вызовом `clean_text(query)`, без временного столбца pandas и `.iloc[0]`.

Мы заменяем пропуск пустой строкой, переводим буквы в нижний регистр, заменяем «ё» на «е», убираем пунктуацию и повторяющиеся пробелы. Регулярное выражение `[^а-яa-z0-9\s]` находит символы, которые не являются буквами, цифрами или пробельными символами. `\s+` находит один или несколько пробельных символов.

К столбцу применяем эту функцию через `.map(clean_text)`: pandas вызывает её для каждого значения. Лемматизацию пока не используем; различия окончаний попробуем обработать символьным поиском.

In [9]:
def clean_text(text):
    """Очищает одну строку: убирает пунктуацию и лишние пробелы.
    Для пропуска возвращает пустую строку.
    """
    if pd.isna(text):
        return ""

    text = text.lower()
    text = text.replace("ё", "е")
    text = re.sub(r"[^а-яa-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


print(clean_text("  РЕМОНТ    телевизоров!!!  "))

ремонт телевизоров


In [10]:
items["clean_title"] = items["item_title_raw"].map(clean_text)
items["clean_parameters"] = items["item_infm_params_text"].map(clean_text)
descriptions = items["item_description_raw"].str.slice(0, 1500).map(clean_text)

# Заголовок повторён дважды, чтобы его слова сильнее влияли на поиск.
items["search_text"] = (
    items["clean_title"] + " " + items["clean_title"] + " "
    + items["clean_parameters"] + " " + descriptions
)

# Для частей слов берём короче: так индекс требует меньше памяти.
items["short_text"] = items["clean_title"] + " " + items["clean_parameters"]
queries["clean_query"] = queries["search_query"].map(clean_text)

del descriptions
items[["item_title_raw", "search_text"]].head(3)

,item_title_raw,search_text
0,Ремонт/выкуп компьют. и ноутбуков с выездом на...,ремонт выкуп компьют и ноутбуков с выездом на ...
1,Обучение ребенка чтению,обучение ребенка чтению обучение ребенка чтени...
2,Афрокудри 5+,афрокудри 5 афрокудри 5 вид услуги красота здо...


## 5. Готовим запросы с известными ответами

Объединяем строки train с одинаковыми пятью условиями поиска. Для каждого запроса получаем набор выбранных объявлений. Например, три выбора `A, B, A` превращаются в `{A, B}`: повторный выбор не увеличивает вес ответа.

Если часть правильных объявлений отсутствует в корпусе, такой запрос исключаем из локальной оценки целиком. Это позволяет проверять поиск только там, где все известные ответы в принципе доступны. **Полученная выборка смещена относительно бенчмарка**: её оценку нельзя выдавать за ожидаемый результат платформы.

In [11]:
groups = train.groupby(search_columns, dropna=False)["item_id"].agg(set)
groups = groups.reset_index(name="relevant_items")

corpus_ids = set(items["item_id"])
can_evaluate = []

for relevant_items in groups["relevant_items"]:
    # Все известные ответы должны существовать в корпусе.
    all_answers_exist = relevant_items.issubset(corpus_ids)
    can_evaluate.append(all_answers_exist)

eligible = groups[can_evaluate].copy()
eligible["clean_query"] = eligible["search_query"].map(clean_text)
eligible = eligible.reset_index(drop=True)

print("Всего групп запросов:", len(groups))
print("Все известные ответы есть в корпусе:", len(eligible))
assert len(eligible) > 0, "Нет запросов для проверки"

Всего групп запросов: 354463
Все известные ответы есть в корпусе: 18436


## 6. Отделяем обучение от проверки

У модели будет три разных набора запросов:

- **train_part** — учим модель отличать выбранные объявления от остальных кандидатов;
- **dev_part** — сравниваем методы и выбираем настройки;
- **test_part** — один раз проверяем уже выбранный вариант.

Одинаковый очищенный текст запроса не попадает в разные части, даже если города или фильтры отличаются. Это более строгая проверка переноса на новые формулировки, чем случайное разделение строк. Сначала разделяем все доступные группы, затем ограничиваем размер каждой части для скорости.

Ни метки dev, ни метки test не используются при обучении бустинга. TF-IDF строится по текстам корпуса: они доступны и при реальном поиске. Это не использование ответов проверочных запросов.

In [12]:
# Отделяем примерно 70% уникальных текстов для обучения.
# Остальные 30% позже разделим на dev и test.
splitter = GroupShuffleSplit(
    n_splits=1,          # Создаём одно разбиение.
    test_size=0.30,      # Доля уникальных текстов во второй части.
    random_state=SEED,   # При повторном запуске разбиение будет таким же.
)

# Одинаковый clean_query всегда попадает в одну часть,
# даже если локации или фильтры у этих запросов отличаются.
# next() достаёт первое разбиение — два массива позиций строк.
train_positions, other_positions = next(
    splitter.split(eligible, groups=eligible["clean_query"])
)

# По полученным позициям выбираем строки исходной таблицы.
train_pool = eligible.iloc[train_positions]
other_pool = eligible.iloc[other_positions]

# Оставшиеся тексты делим пополам между dev и test.
# В итоге получаем примерно 70% / 15% / 15% уникальных текстов.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=SEED,
)

# Эти позиции относятся уже к таблице other_pool.
dev_positions, test_positions = next(
    splitter.split(other_pool, groups=other_pool["clean_query"])
)

dev_pool = other_pool.iloc[dev_positions]
test_pool = other_pool.iloc[test_positions]

# Для первого эксперимента используем только часть доступных строк.
# min() не позволяет запросить больше строк, чем есть в таблице.
# sample() случайно выбирает строки без повторов.
train_size = min(TRAIN_SIZE, len(train_pool))
train_part = train_pool.sample(n=train_size, random_state=SEED)

dev_size = min(DEV_SIZE, len(dev_pool))
dev_part = dev_pool.sample(n=dev_size, random_state=SEED)

test_size = min(TEST_SIZE, len(test_pool))
test_part = test_pool.sample(n=test_size, random_state=SEED)

# После выбора заново нумеруем строки: 0, 1, 2, ...
# Старые номера не сохраняем отдельным столбцом.
train_part = train_part.reset_index(drop=True)
dev_part = dev_part.reset_index(drop=True)
test_part = test_part.reset_index(drop=True)

# Собираем уникальные очищенные тексты каждой части.
train_texts = set(train_part["clean_query"])
dev_texts = set(dev_part["clean_query"])
test_texts = set(test_part["clean_query"])

# Проверяем, что одинаковых текстов между частями нет.
# Если пересечение есть, assert остановит выполнение.
assert train_texts.isdisjoint(dev_texts), "Есть общие тексты в train и dev"
assert train_texts.isdisjoint(test_texts), "Есть общие тексты в train и test"
assert dev_texts.isdisjoint(test_texts), "Есть общие тексты в dev и test"

# Показываем число выбранных строк с условиями поиска.
print("Обучение:", len(train_part))
print("Выбор метода:", len(dev_part))
print("Финальная проверка:", len(test_part))

Обучение: 2500
Выбор метода: 500
Финальная проверка: 500


## 7. Проверяем роль локации и категории

Используем только обучающую часть. Для каждого известного выбранного объявления сравниваем его город и категорию с условиями запроса.

Даже высокая доля совпадений не доказывает, что можно ввести жёсткий фильтр: услуги могут предоставляться в соседнем городе или дистанционно. Категория `0` в запросе рассматривается как отсутствие конкретного ограничения, а не как отдельный тип услуг. ID городов и категорий — коды: их нельзя сравнивать по величине как расстояния.

In [13]:
items_by_id = items.set_index("item_id")
same_locations = []
same_categories = []

for _, row in train_part.iterrows():
    for item_id in row["relevant_items"]:
        item = items_by_id.loc[item_id]
        same_locations.append(row["search_location_id"] == item["item_location_id"])
        if row["search_category"] != 0:
            same_categories.append(row["search_category"] == item["item_category_id"])

print("Совпадение локации у выбранных объявлений (%):", round(np.mean(same_locations) * 100, 2))
print("Совпадение указанной категории (%):", round(np.mean(same_categories) * 100, 2))

Совпадение локации у выбранных объявлений (%): 80.58
Совпадение указанной категории (%): 100.0


## 8. Два индекса TF-IDF

### Поиск по целым словам

`fit_transform` изучает словарь корпуса и создаёт числовую матрицу: строки — объявления, столбцы — слова. Разреженное хранение позволяет не хранить большинство нулей.

- `min_df=2`: слово должно встретиться минимум в двух объявлениях;
- `max_features=100000`: ограничиваем словарь для экономии памяти;
- `sublinear_tf=True`: повторения слова увеличивают его вес медленнее, по формуле `1 + log(число повторов)`;
- `float32`: используем менее затратный тип чисел.

Это те же основные настройки, что и в исходном базовом решении. TF-IDF по умолчанию нормирует длину векторов. Поэтому их скалярное произведение даёт косинусное сходство, а не вероятность выбора.

In [14]:
word_vectorizer = TfidfVectorizer(
    min_df=2,
    max_features=100000,
    sublinear_tf=True,
    dtype=np.float32,
)
word_vectors = word_vectorizer.fit_transform(items["search_text"])
print("Размер матрицы слов:", word_vectors.shape)

Размер матрицы слов: (189212, 100000)


### Поиск по частям слов

Слова «телевизор» и «телевизоров» различаются целиком, но имеют много одинаковых фрагментов. Символьный TF-IDF использует фрагменты длиной от 3 до 5 символов. Он помогает с окончаниями и некоторыми опечатками, но сам по себе не понимает синонимы.

`char_wb` выделяет фрагменты внутри слов с учётом границ слова. Индекс строим только по заголовку и параметрам: длинные описания сделали бы его значительно больше. Это компромисс памяти и полноты, который оцениваем экспериментом.

In [15]:
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    max_features=150000,
    sublinear_tf=True,
    dtype=np.float32,
)
char_vectors = char_vectorizer.fit_transform(items["short_text"])
print("Размер матрицы частей слов:", char_vectors.shape)

Размер матрицы частей слов: (189212, 150000)


## 9. Как получаем расширенный набор кандидатов

Для каждого запроса объединяем четыре списка:

1. До 100 совпадений по словам по всему корпусу.
2. До 100 совпадений по частям слов по всему корпусу.
3. До 100 совпадений по словам в указанной локации.
4. До 100 совпадений по частям слов в указанной локации.

После удаления повторов остаётся **до 400 кандидатов**. Локальные списки добавляют объявления, которые могли потеряться среди похожих текстов из других городов. Глобальные списки сохраняют возможность найти услугу в другой локации. Если текст пустой, ищем по тексту фильтров. Если нет ни одного текстового совпадения, возвращаем пустой список — это допустимо по формату, хотя ограничивает Recall.

Сначала определим функцию, которая возвращает позиции лучших положительных оценок. Стабильная сортировка сохраняет исходный порядок при равных оценках, поэтому результат воспроизводим.

In [16]:
item_ids = items["item_id"].to_numpy()
item_locations = items["item_location_id"].to_numpy()
item_categories = items["item_category_id"].to_numpy()

# Для каждого города заранее запоминаем позиции объявлений.
location_positions = items.groupby("item_location_id").indices


def top_positions(scores, limit):
    """Возвращает до limit позиций с наибольшими положительными оценками.
    scores — массив оценок. Результат содержит номера позиций, не item_id.
    """
    # Сначала убираем оценки без совпадений.
    positive_mask = scores > 0
    positions = np.flatnonzero(positive_mask)
    positive_scores = scores[positions]

    # Минус даёт сортировку от большей оценки к меньшей.
    order = np.argsort(-positive_scores, kind="stable")
    best_order = order[:limit]
    return positions[best_order]

### Почему запросы считаются небольшими пакетами

Формула остаётся той же: `векторы запросов @ матрица объявлений.T`. В результате строка соответствует запросу, столбец — объявлению. Один пакет содержит 32 запроса. Это быстрее повторного вызова для каждого запроса и не требует хранить огромную плотную матрицу для всех запросов сразу.

В `records` сохраняем только найденных кандидатов и их оценки. Разметка `relevant_items` при поиске не используется: функция одинаково работает для обучения и бенчмарка.

In [17]:
def retrieve_candidates(table):
    """Находит до 400 кандидатов для каждого запроса из table.
    Использует подготовленные выше индексы TF-IDF и локации объявлений.
    Возвращает список словарей с позициями кандидатов и оценками сходства.
    Порядок словарей совпадает с порядком строк table.
    """
    records = []

    for start in range(0, len(table), BATCH_SIZE):
        batch = table.iloc[start:start + BATCH_SIZE]
        texts = []

        for _, row in batch.iterrows():
            text = clean_text(row["search_query"])
            if text == "":
                text = clean_text(row["search_infm_params_text"])
            texts.append(text)

        query_words = word_vectorizer.transform(texts)
        query_chars = char_vectorizer.transform(texts)
        # Строка матрицы содержит оценки одного запроса.
        word_scores = query_words @ word_vectors.T
        word_scores = word_scores.toarray()
        char_scores = query_chars @ char_vectors.T
        char_scores = char_scores.toarray()

        for number in range(len(batch)):
            row = batch.iloc[number]
            words = word_scores[number]
            chars = char_scores[number]

            word_best = top_positions(words, CHANNEL_SIZE)
            char_best = top_positions(chars, CHANNEL_SIZE)
            local = location_positions.get(row["search_location_id"], np.array([], dtype=int))
            local_word_best = local[top_positions(words[local], CHANNEL_SIZE)]
            local_char_best = local[top_positions(chars[local], CHANNEL_SIZE)]

            positions = np.unique(np.concatenate([
                word_best, char_best, local_word_best, local_char_best,
            ]))

            records.append({
                "positions": positions,
                "word_score": words[positions],
                "char_score": chars[positions],
                "baseline": word_best[:TOP_K],
            })

        if (start // BATCH_SIZE + 1) % 20 == 0:
            print("Обработано запросов:", min(start + BATCH_SIZE, len(table)))

    return records

## 10. Качество базового поиска и полнота кандидатов

Сначала считаем кандидатов для обучающей части и dev. Финальную test-часть пока не используем.

`Recall кандидатов` показывает, какая доля правильных объявлений вообще попала в расширенный список. Это верхняя граница достижимого Recall@50 при данном наборе кандидатов. Она может быть выше, чем реальное качество отбора: модель ещё должна выбрать нужные объявления из списка.

Базовый метод берёт первые 50 совпадений только по словам. Новый контрольный набор отличается от старых 500 запросов, поэтому прежнее число 0.2265 напрямую с новыми оценками не сравниваем.

In [18]:
train_records = retrieve_candidates(train_part)
dev_records = retrieve_candidates(dev_part)


def recall_for_query(predicted_positions, relevant_items):
    """Считает долю найденных правильных ответов одного запроса.
    predicted_positions — позиции объявлений в корпусе.
    relevant_items — непустое множество правильных item_id.
    """
    predicted_ids = set(item_ids[predicted_positions])
    found = predicted_ids.intersection(relevant_items)
    return len(found) / len(relevant_items)


def evaluate_lists(table, predictions):
    """Возвращает средний Recall и массив оценок отдельных запросов.
    table содержит правильные ответы в relevant_items.
    predictions содержит списки позиций объявлений в порядке строк table.
    """
    recalls = []
    for relevant_items, predicted_positions in zip(table["relevant_items"], predictions):
        recalls.append(recall_for_query(predicted_positions, relevant_items))
    return float(np.mean(recalls)), np.array(recalls)


dev_baseline, _ = evaluate_lists(dev_part, [record["baseline"] for record in dev_records])
dev_ceiling, _ = evaluate_lists(dev_part, [record["positions"] for record in dev_records])
print("Базовый Recall@50 на dev:", round(dev_baseline, 4))
print("Recall расширенного списка на dev:", round(dev_ceiling, 4))

Обработано запросов: 640
Обработано запросов: 1280
Обработано запросов: 1920
Базовый Recall@50 на dev: 0.294
Recall расширенного списка на dev: 0.865


## 11. Простой отбор без обучения

Складываем оценки поиска по словам и частям слов. Для объявления из указанного города умножаем сумму на `1 + location_weight`. При весе `0` город ничего не меняет, при весе `1` совпадение города удваивает оценку.

Сравним четыре заранее выбранных веса: 0, 0.5, 1 и 2. Это эвристика: коэффициент не выдаём за вероятность, а выбираем по Recall@50 на dev. Категорию используем далее как признак модели, не вводя жёсткого отсева.

In [19]:
def hybrid_predictions(table, records, location_weight):
    """Выбирает до 50 объявлений по текстовому сходству и локации.
    table — запросы, records — их кандидаты в том же порядке.
    location_weight задаёт дополнительный вес совпадающей локации.
    Возвращает список массивов позиций объявлений.
    """
    predictions = []

    for number in range(len(table)):
        row = table.iloc[number]
        record = records[number]
        positions = record["positions"]
        same_location = item_locations[positions] == row["search_location_id"]
        scores = record["word_score"] + record["char_score"]
        scores = scores * (1 + location_weight * same_location)
        order = np.argsort(-scores, kind="stable")[:TOP_K]
        predictions.append(positions[order])

    return predictions


results = [{"method": "word_tfidf", "dev_recall_at_50": dev_baseline}]
hybrid_weights = {}

for weight in [0.0, 0.5, 1.0, 2.0]:
    name = "hybrid_" + str(weight)
    hybrid_weights[name] = weight
    predictions = hybrid_predictions(dev_part, dev_records, weight)
    score, _ = evaluate_lists(dev_part, predictions)
    results.append({"method": name, "dev_recall_at_50": score})

pd.DataFrame(results)

,method,dev_recall_at_50
0,word_tfidf,0.294000
1,hybrid_0.0,0.293000
2,hybrid_0.5,0.549000
3,hybrid_1.0,0.690667
4,hybrid_2.0,0.756000


## 12. Признаки для обучаемой модели

Теперь для каждой пары «запрос — кандидат» создаём небольшую числовую строку. Модель получает:

- оценки совпадения по словам и частям слов, их сумму и произведение;
- совпадение города, совпадение категории и наличие ограничения категории;
- длину запроса и флаг поиска с доставкой;
- совпадение слов фильтра с параметрами объявления;
- рейтинг, число отзывов, цену и признаки отсутствующих значений;
- возможность звонка и отправки сообщения.

Для цены и числа отзывов используем `log(1 + значение)`, чтобы большие числа не отличались от остальных на много порядков. Пропуск цены не означает бесплатную услугу: отдельный признак сообщает модели о пропуске. Пропуск рейтинга также отмечается отдельно.

Мы не передаём модели `query_id`, `item_id` и числовые коды городов как величины. Деревья учатся общим закономерностям. Географическое расстояние пока не используем: для запроса нет готовых координат, а восстанавливать их по объявлениям означало бы добавлять отдельные предположения.

In [20]:
item_ratings = pd.to_numeric(items["item_rating"], errors="coerce").to_numpy(dtype=float)
item_reviews = pd.to_numeric(items["item_rating_reviews_count"], errors="coerce").to_numpy(dtype=float)
item_prices = pd.to_numeric(items["item_price"], errors="coerce").to_numpy(dtype=float)
phone_hidden = items["item_is_phone_hidden"].fillna(False).to_numpy(dtype=float)
message_forbidden = items["item_is_message_forbidden"].fillna(False).to_numpy(dtype=float)
parameter_tokens = [set(text.split()) for text in items["clean_parameters"]]

feature_names = [
    "word_score", "char_score", "score_sum", "score_product",
    "same_location", "same_category", "category_specified",
    "query_length", "delivery_search", "filter_overlap",
    "rating", "rating_missing", "log_reviews", "reviews_missing",
    "log_price", "price_missing", "phone_hidden", "message_forbidden",
]


def make_features(row, record):
    """Рассчитывает 18 признаков для кандидатов одного запроса.
    row — строка запроса, record — словарь его кандидатов.
    Возвращает матрицу: строка на кандидата, столбец на признак.
    Порядок столбцов соответствует feature_names.
    Свойства объявлений берутся из массивов, подготовленных выше.
    """
    positions = record["positions"]
    count = len(positions)
    words = record["word_score"]
    chars = record["char_score"]
    same_location = item_locations[positions] == row["search_location_id"]
    category_specified = pd.notna(row["search_category"]) and row["search_category"] != 0
    same_category = (item_categories[positions] == row["search_category"]) & category_specified

    filter_words = set(clean_text(row["search_infm_params_text"]).split())
    overlap = []
    for position in positions:
        common = filter_words.intersection(parameter_tokens[position])
        overlap.append(len(common) / max(1, len(filter_words)))

    ratings = item_ratings[positions]
    reviews = item_reviews[positions]
    prices = item_prices[positions]
    delivery = row["search_is_delivery_search"]
    if pd.isna(delivery):
        delivery = 0

    # Собираем столбцы в порядке feature_names.
    features = np.column_stack([
        words, chars, words + chars, words * chars,
        same_location, same_category, np.full(count, category_specified),
        np.full(count, len(clean_text(row["search_query"]).split())),
        np.full(count, delivery), overlap,
        np.nan_to_num(ratings), np.isnan(ratings),
        np.log1p(np.maximum(np.nan_to_num(reviews), 0)), np.isnan(reviews),
        np.log1p(np.maximum(np.nan_to_num(prices), 0)), np.isnan(prices),
        phone_hidden[positions], message_forbidden[positions],
    ])
    return features.astype(np.float32)

## 13. Формируем примеры для обучения

Для каждой обучающей группы найденный кандидат получает метку `1`, если он есть среди известных выбранных объявлений, и `0` в остальных случаях. Нули — **условные отрицательные примеры**: отсутствие выбора не доказывает нерелевантность. Мы не имеем логов показов, поэтому полностью устранить это ограничение нельзя.

Сохраняем все найденные положительные примеры и случайно выбираем до 80 остальных кандидатов на запрос. Это сокращает время обучения. Если ни один правильный ответ не найден, запрос пропускаем при обучении отбора, но не при оценке поиска: оценка такого запроса остаётся нулевой.

Веса устроены так, чтобы суммарный вес положительных и отрицательных примеров внутри одного запроса был одинаковым. Это приближает обучение к задаче выбора внутри запроса, но модель всё равно оптимизирует классификационную ошибку, а не Recall@50 напрямую. Поэтому окончательный выбор делаем по Recall@50.

In [21]:
rng = np.random.default_rng(SEED)
feature_parts = []
label_parts = []
weight_parts = []
skipped_queries = 0

for number in range(len(train_part)):
    row = train_part.iloc[number]
    record = train_records[number]
    positions = record["positions"]
    # Метка 0 означает отсутствие записанного выбора, а не плохое объявление.
    labels = []
    for position in positions:
        was_selected = item_ids[position] in row["relevant_items"]
        labels.append(was_selected)
    labels = np.array(labels, dtype=bool)
    positive = np.flatnonzero(labels)
    negative = np.flatnonzero(~labels)

    if len(positive) == 0 or len(negative) == 0:
        skipped_queries += 1
        continue

    negative = rng.choice(negative, size=min(80, len(negative)), replace=False)
    selected = np.concatenate([positive, negative])
    features = make_features(row, record)
    weights = np.concatenate([
        np.full(len(positive), 0.5 / len(positive)),
        np.full(len(negative), 0.5 / len(negative)),
    ])

    feature_parts.append(features[selected])
    label_parts.append(labels[selected].astype(int))
    weight_parts.append(weights)

assert feature_parts, "Нет обучающих пар после поиска кандидатов"
X_train = np.concatenate(feature_parts)
y_train = np.concatenate(label_parts)
sample_weights = np.concatenate(weight_parts)
sample_weights = sample_weights / sample_weights.mean()

print("Размер обучающей таблицы:", X_train.shape)
print("Положительных примеров:", y_train.sum())
print("Пропущено запросов при обучении:", skipped_queries)

Размер обучающей таблицы: (176650, 18)
Положительных примеров: 2250
Пропущено запросов при обучении: 320


## 14. Обучаем градиентный бустинг

Бустинг последовательно строит небольшие деревья решений. Каждое следующее дерево помогает исправлять ошибки предыдущих. Для табличных признаков это подходящий базовый обучаемый метод.

Используем `HistGradientBoostingClassifier` из scikit-learn, который уже входит в окружение проекта. Это альтернатива обсуждавшемуся CatBoost: отдельную библиотеку устанавливать не нужно.

Фиксируем настройки заранее: 120 итераций, не более 15 листьев в дереве, шаг обучения 0.08. Раннюю остановку отключаем, чтобы библиотека не делила строки одной группы случайно между обучением и внутренней проверкой. Гиперпараметры по test не подбираем. Ограничение числа потоков до 4 помогает избежать лишней нагрузки на ноутбук.

Выход `predict_proba` используется как оценка для сортировки. Из-за отбора отрицательных примеров и весов его нельзя считать откалиброванной вероятностью реального выбора пользователем.

In [22]:
model = HistGradientBoostingClassifier(
    max_iter=120,
    max_leaf_nodes=15,
    learning_rate=0.08,
    min_samples_leaf=30,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=SEED,
)

with threadpool_limits(limits=4):
    model.fit(X_train, y_train, sample_weight=sample_weights)


def model_predictions(table, records):
    """Выбирает до 50 кандидатов с помощью обученной модели model.
    table и records содержат запросы и их кандидатов в одном порядке.
    Возвращает список массивов позиций объявлений в корпусе.
    """
    predictions = []
    with threadpool_limits(limits=4):
        for number in range(len(table)):
            row = table.iloc[number]
            record = records[number]
            positions = record["positions"]

            if len(positions) == 0:
                predictions.append(positions)
                continue

            features = make_features(row, record)
            # Берём оценки класса 1: известный выбор объявления.
            class_scores = model.predict_proba(features)
            scores = class_scores[:, 1]
            
            order = np.argsort(-scores, kind="stable")[:TOP_K]
            predictions.append(positions[order])
    return predictions


dev_model_predictions = model_predictions(dev_part, dev_records)
dev_model_score, _ = evaluate_lists(dev_part, dev_model_predictions)
results.append({"method": "gradient_boosting", "dev_recall_at_50": dev_model_score})

comparison = pd.DataFrame(results).sort_values("dev_recall_at_50", ascending=False)
chosen_method = comparison.iloc[0]["method"]
comparison.to_csv("outputs/dev_comparison.csv", index=False)
print("Выбранный на dev метод:", chosen_method)
comparison

Выбранный на dev метод: gradient_boosting


,method,dev_recall_at_50
5,gradient_boosting,0.788000
4,hybrid_2.0,0.756000
3,hybrid_1.0,0.690667
2,hybrid_0.5,0.549000
0,word_tfidf,0.294000
1,hybrid_0.0,0.293000


## 15. Итоговая проверка на отложенных запросах

Метод уже выбран по dev. Теперь считаем test один раз и сравниваем выбранный вариант с исходным поиском по словам. Если результат на test хуже, это честно попадёт в выводы; автоматически переключать метод по test нельзя, иначе test превратится в ещё одну выборку для подбора.

Дополнительно оцениваем неопределённость разницы методом bootstrap: многократно выбираем с повторениями тексты запросов и пересчитываем разницу метрик. Группировка по текстам сохраняет зависимость между одинаковыми формулировками в разных локациях. Интервал приблизительный и относится только к нашему ограниченному контрольному набору.

In [23]:
def selected_predictions(table, records):
    """Применяет метод, выбранный на dev, к запросам и их кандидатам.
    Использует переменную chosen_method, заданную выше.
    Возвращает список массивов позиций объявлений.
    """
    if chosen_method == "word_tfidf":
        return [record["baseline"] for record in records]
    if chosen_method == "gradient_boosting":
        return model_predictions(table, records)
    return hybrid_predictions(table, records, hybrid_weights[chosen_method])


test_records = retrieve_candidates(test_part)
test_baseline_predictions = [record["baseline"] for record in test_records]
test_selected_predictions = selected_predictions(test_part, test_records)

test_baseline, baseline_recalls = evaluate_lists(test_part, test_baseline_predictions)
test_selected, selected_recalls = evaluate_lists(test_part, test_selected_predictions)
test_ceiling, _ = evaluate_lists(test_part, [record["positions"] for record in test_records])

difference = selected_recalls - baseline_recalls
bootstrap_groups = pd.DataFrame({
    "text": test_part["clean_query"], "difference": difference,
}).groupby("text")["difference"].agg(["sum", "count"])
rng = np.random.default_rng(SEED)
bootstrap_differences = []

for _ in range(2000):
    sample = rng.integers(0, len(bootstrap_groups), len(bootstrap_groups))
    sampled_groups = bootstrap_groups.iloc[sample]
    bootstrap_differences.append(sampled_groups["sum"].sum() / sampled_groups["count"].sum())

interval = np.quantile(bootstrap_differences, [0.025, 0.975])
print("Базовый Recall@50 на test:", round(test_baseline, 4))
print("Выбранный метод на test:", round(test_selected, 4))
print("Recall расширенного списка на test:", round(test_ceiling, 4))
print("Разница:", round(test_selected - test_baseline, 4))
print("Приблизительный 95% интервал разницы:", interval.round(4))

Базовый Recall@50 на test: 0.2595
Выбранный метод на test: 0.7445
Recall расширенного списка на test: 0.83
Разница: 0.485
Приблизительный 95% интервал разницы: [0.4341 0.5362]


## 16. Разбираем ошибки

Если правильного объявления нет даже в расширенном наборе, улучшать нужно поиск кандидатов. Если оно есть в наборе, но не входит в выбранные 50, ошибка возникла при отборе. По сохранённой таблице можно различить эти два случая.

Примеры test используются для объяснения результата. Новые изменения после просмотра этих ошибок потребуют новой независимой проверки. Не восстанавливаем ответы бенчмарка вручную и не пишем правил под конкретные query_id.

In [24]:
error_rows = []

for number, row in test_part.iterrows():
    if selected_recalls[number] == 1:
        continue
    relevant = row["relevant_items"]
    retrieved = set(item_ids[test_records[number]["positions"]])
    selected = set(item_ids[test_selected_predictions[number]])
    missed = sorted(relevant - selected)
    titles = items_by_id.loc[missed, "item_title_raw"].tolist()
    error_rows.append({
        "query": row["search_query"],
        "location": row["search_location_id"],
        "recall_at_50": selected_recalls[number],
        "relevant_count": len(relevant),
        "relevant_in_candidates": len(relevant.intersection(retrieved)),
        "missed_titles": " | ".join(titles),
    })

errors = pd.DataFrame(error_rows)
errors.to_csv("outputs/test_errors.csv", index=False)
errors.head(10)

,query,location,recall_at_50,relevant_count,relevant_in_candidates,missed_titles
0,обклейка холодильника пленкой,637640,0.0,1,1,Оклейка плëнкой. Интерьер/Экстерьер
1,печать книги,653240,0.0,1,0,Контрактная печать книг от 1 экз. кбс / шкс / 7БЦ
2,блокировка айфона,107621,0.0,1,0,Ремонт iPhone iPad iMac Macbook Apple Айфон аймак
3,вспашка минитрактором,623430,0.0,1,0,Планирование участка / Выравнивание участка
4,татуаж губ,107620,0.0,1,0,Перманентный макияж бровей губ и век татуаж
5,расслабляющий массаж,107621,0.0,1,0,Массаж для полной перезагрузки
6,фотостудия без фотографа,653240,0.0,1,1,"Аренда фотостудии, циклорама"
7,вело ремонт,641510,0.0,1,0,Ремонт велосипеда (сезонное обслуживание)
8,карго доставка из китая розница,621540,0.0,1,1,Байер. Доставка из Китая под ключ (карго/розница)
9,модель на педикюр,107620,0.0,1,0,Маникюр педикюр


## 17. Готовим ответы для бенчмарка

Применяем метод, выбранный на dev. Для воспроизводимости сохраняем ту же обученную модель: после test не переобучаем её на контрольных ответах. Идентификаторы берём только из исходного корпуса и записываем без изменения регистра.

In [25]:
benchmark_records = retrieve_candidates(queries)
benchmark_predictions = selected_predictions(queries, benchmark_records)

answer_rows = []
for positions in benchmark_predictions:
    answer_rows.append(" ".join(item_ids[positions]))

answer = pd.DataFrame({
    "query_id": queries["query_id"].tolist(),
    "answer": answer_rows,
})
answer.head(3)

Обработано запросов: 640
Обработано запросов: 1280
Обработано запросов: 1920


,query_id,answer
0,70DfDUpwjxB4lzFd,355392014208b7bf 73ba3172bc67d2dd fa4ba2c2cf71...
1,JTrdTaZJvSiLPkXj,dfa239d031b11832 2f09017b0819844e 62a6515c16d0...
2,LZCZNoVG4AFUkVRJ,c5d21c0e8c1cdb2d d8fce513e4f000a7 10eb6d6da323...


## 18. Проверяем формат и сохраняем answer.csv

Проверяем набор запросов, число объявлений, повторы и принадлежность корпусу. `join` выше соединяет item_id ровно одним пробелом. `index=False` не позволяет pandas добавить лишнюю колонку с номерами строк. После сохранения читаем файл как строки и проверяем, что его содержимое не изменилось.

In [26]:
assert list(answer.columns) == ["query_id", "answer"]
assert answer["query_id"].is_unique
assert len(answer) == len(queries)
assert set(answer["query_id"]) == set(queries["query_id"])

for text in answer["answer"]:
    candidates = text.split()
    assert len(candidates) <= TOP_K
    assert len(candidates) == len(set(candidates))
    assert set(candidates).issubset(corpus_ids)
    assert all(re.fullmatch("[0-9a-f]{16}", item_id) for item_id in candidates)

answer.to_csv("answer.csv", index=False, encoding="utf-8")
saved_answer = pd.read_csv("answer.csv", dtype=str, keep_default_na=False)
pd.testing.assert_frame_equal(answer, saved_answer)

empty_answers = int((answer["answer"] == "").sum())
print("Файл сохранён. Запросов:", len(answer))
print("Пустых ответов:", empty_answers)

Файл сохранён. Запросов: 2452
Пустых ответов: 0


## 19. Сохраняем результаты и формулируем выводы

Числа ниже получены текущим запуском, а не записаны заранее. Модель сохраняем для изучения; для полного повторения эксперимента достаточно выполнить ноутбук. Файлы joblib следует загружать только из доверенного источника.

Главные ограничения: доступна лишь часть обучающих ответов в корпусе; выбранные пользователем объявления не исчерпывают все релевантные; train не содержит информацию о показах; нет временных меток для временного разбиения; поиск по частям слов не решает полноценное сопоставление синонимов; эксперимент ограничен 2500 обучающими запросами. Ниже приводится локальная оценка, а не результат скрытого бенчмарка.

In [27]:
import joblib

joblib.dump({"model": model, "feature_names": feature_names}, "outputs/selector.joblib")

metrics = {
    "seed": SEED,
    "train_queries": len(train_part),
    "dev_queries": len(dev_part),
    "test_queries": len(test_part),
    "eligible_queries": len(eligible),
    "train_pairs": len(y_train),
    "skipped_train_queries": skipped_queries,
    "chosen_method": chosen_method,
    "dev_results": results,
    "dev_candidate_recall": dev_ceiling,
    "test_baseline_recall_at_50": test_baseline,
    "test_selected_recall_at_50": test_selected,
    "test_candidate_recall": test_ceiling,
    "test_difference_95_interval": interval.tolist(),
    "benchmark_queries": len(answer),
    "empty_answers": empty_answers,
    "elapsed_seconds": round(time.time() - start_time, 1),
}
with open("outputs/metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)

print("По dev выбран:", chosen_method)
print("На test базовый метод:", round(test_baseline, 4))
print("На test выбранный метод:", round(test_selected, 4))
print("Изменение в процентных пунктах:", round(100 * (test_selected - test_baseline), 2))
print("Объявления вне расширенного списка ограничивают Recall уровнем:", round(test_ceiling, 4))
if interval[0] > 0:
    print("На этой выборке интервал разницы положительный: улучшение устойчиво к bootstrap-перевыборке.")
else:
    print("Интервал включает ноль или лежит ниже: уверенно утверждать улучшение по этой проверке нельзя.")
print("Время выполнения, секунд:", metrics["elapsed_seconds"])

По dev выбран: gradient_boosting
На test базовый метод: 0.2595
На test выбранный метод: 0.7445
Изменение в процентных пунктах: 48.5
Объявления вне расширенного списка ограничивают Recall уровнем: 0.83
На этой выборке интервал разницы положительный: улучшение устойчиво к bootstrap-перевыборке.
Время выполнения, секунд: 2416.3


## Выводы выполненного эксперимента

На dev лучшим оказался градиентный бустинг: **Recall@50 = 0,7880**. Его применили к независимой test-части без изменения настроек.

| Метод на test | Recall@50 |
|---|---:|
| Базовый поиск по словам | 0.2595 |
| Обучаемый отбор | 0.7445 |
| Полнота расширенного списка (до 400) | 0.8300 |

Улучшение относительно базового поиска составляет **48,5 процентного пункта**. Приблизительный 95% интервал разницы: **[0,4341; 0,5362]**. Это результат локальной проверки, а не скрытого бенчмарка.

Само сложение словарного и символьного сходства без учёта локации не улучшило baseline на dev. Большой вклад дал учёт локации; бустинг добавил 3,2 процентного пункта к лучшей проверенной эвристике на dev.

У 85 test-запросов правильных объявлений нет в расширенном списке; ещё у 44 запросов все известные правильные объявления доступны отбору, но не все входят в финальные 50. Поэтому улучшать можно и поиск кандидатов, и отбор, причём полнота первого этапа ограничивает итоговое качество.

Создан `answer.csv` для **2452 запросов**, пустых ответов **0**. Полное выполнение заняло **35.3 минуты** на текущем компьютере. Исходные данные не изменены; все вычисления локальные.

Подробные объяснения результатов и ограничений находятся в [REPORT.md](REPORT.md). Новые настройки после просмотра test следует оценивать на новой независимой выборке.


## Что можно улучшать дальше

1. Расширить обучение и оценку на большее число запросов, учитывая затраты времени и памяти.
2. Если нужные объявления не попадают в расширенный список — пробовать BM25, нормализацию слов или локальные семантические векторы.
3. Если объявления есть в списке, но отбор их теряет — улучшать признаки, обучение и оценивать CatBoost/модели ранжирования на новом корректном разбиении.
4. Проверить общий итог через систему задания. Семь отправок не заменяют локальную проверку; частые изменения под результат платформы могут привести к переобучению на бенчмарк.

Основной результат работы — не обещание идеального поиска, а воспроизводимый эксперимент: есть исходный метод, обучаемое улучшение, независимая проверка, анализ ограничений и валидный файл ответа.